In [0]:
#Take the schema from existing raw table
user_schema = spark.table("raw.app.users").schema

#Read JSON files from S3 as a stream
df_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .schema(user_schema)
        .load("/Volumes/hinge_dev/s3/hinge_datalake/raw/users/")
)

# Incremental load with checkpoint + stop after processing new files
(
    df_stream.writeStream
        .format("delta")
        .option("checkpointLocation",
                "/Volumes/hinge_dev/s3/hinge_datalake/checkpoints/users")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable("raw.app.users")
)
